# 常暗之厢 — 四步渐进式解析（分步调试版）\n\n**每步单独执行**，中间结果、完整 prompt 和 LLM 响应全部保存到 `data/debug/` 目录。\n\n**流程**：\n1. Step 1：名称固化 + 精修模组（2 calls 并行）\n2. Step 2：内容生成 — interactions 先跑 → events + auto_triggers + L1 + L3 并行\n3. Step 3：依赖解析 + 交叉核对（2 calls 串行）\n4. Step 4：Library 匹配\n\n**产物**：`data/debug/<timestamp>/` 下每步一个子文件夹，每个 LLM 调用 `prompt.txt` + `response.json`

In [ ]:
# ═══════════════════════════════════════════════════════════════\n# 导入 & 环境配置\n# ═══════════════════════════════════════════════════════════════\nimport sys, json, os, re\nfrom datetime import datetime\nfrom concurrent.futures import ThreadPoolExecutor\n\nsys.path.insert(0, \"../src\")\n\nfrom utils import parser, estimate_and_truncate_context\nfrom llm import call_deepseek\nfrom module_designer import (\n    validate_all, save_pipeline_result,\n    # Prompt builders\n    build_step1a_prompt, build_step1b_prompt,\n    build_step2a_prompt, build_step2b_events_prompt, build_step2b_at_prompt,\n    build_step2c_l1_prompt, build_step2c_l3_prompt,\n    build_step3a_prompt, build_step3b_prompt, build_step4_prompt,\n    # Parsers (for cross_validate)\n    _is_valid_json_output, _with_fallback,\n)\nfrom module_designer.layered_pipeline import cross_validate_layers, CrossRefReport\nfrom module_designer.layered_parser import (\n    STEP1A_SYSTEM, STEP1B_SYSTEM,\n    STEP2A_SYSTEM, STEP2B_EVENTS_SYSTEM, STEP2B_AT_SYSTEM,\n    STEP2C_L1_SYSTEM, STEP2C_L3_SYSTEM,\n    STEP3A_SYSTEM, STEP3B_SYSTEM, STEP4_SYSTEM,\n)\nfrom library import WeaponLibrary, EnemyLibrary\n\nprint(\"模块导入完成\")

In [ ]:
# ═══════════════════════════════════════════════════════════════\n# 加载模组 & 初始化库 & 创建调试目录\n# ═══════════════════════════════════════════════════════════════\n\n# 加载源文档\ncontent = parser(\"../常暗之厢（7版规则，简体修正版）.docx\")\ncontent = estimate_and_truncate_context(content)\nprint(f\"源文档: {len(content)} 字符 (~{len(content)//2} tokens)\")\n\n# 初始化武器/敌人库\nwl = WeaponLibrary(); wl.load_core()\nel = EnemyLibrary(); el.load_core()\nprint(f\"武器: {[w.name for w in wl.list_all()]}\")\nprint(f\"敌人: {[e.name for e in el.list_all()]}\")\n\n# 创建调试输出目录 (临时产物，不放在 data/modules/)\nTIMESTAMP = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\nDEBUG_ROOT = f\"../data/debug/{TIMESTAMP}\"\nos.makedirs(DEBUG_ROOT, exist_ok=True)\nprint(f\"\\n调试产物目录: {DEBUG_ROOT}/\")"

In [ ]:
# ═══════════════════════════════════════════════════════════════\n# 辅助函数：保存 LLM 调用的 prompt + response\n# ═══════════════════════════════════════════════════════════════\n\ndef save_llm_call(step_name, call_name, prompt_text, system_text, response, is_json):\n    \"\"\"\n    将一个 LLM 调用的 prompt 和 response 保存到调试目录。\n    路径: {DEBUG_ROOT}/{step_name}/{call_name}/\n      prompt.txt   — 完整 prompt（含 system）\n      response.json / response.txt — LLM 返回\n    \"\"\"\n    call_dir = os.path.join(DEBUG_ROOT, step_name, call_name)\n    os.makedirs(call_dir, exist_ok=True)\n    \n    # 保存 prompt\n    with open(os.path.join(call_dir, \"prompt.txt\"), \"w\", encoding=\"utf-8\") as f:\n        if system_text:\n            f.write(f\"=== SYSTEM ===\\n{system_text}\\n\\n=== USER PROMPT ===\\n{prompt_text}\")\n        else:\n            f.write(prompt_text)\n    \n    # 保存 response\n    ext = \"json\" if is_json else \"txt\"\n    with open(os.path.join(call_dir, f\"response.{ext}\"), \"w\", encoding=\"utf-8\") as f:\n        if is_json:\n            json.dump(response, f, ensure_ascii=False, indent=2)\n        else:\n            f.write(str(response))\n\ndef do_json_call(step_name, call_name, prompt_fn, *args, system_prompt=\"\", **kwargs):\n    \"\"\"\n    执行一次 JSON 模式 LLM 调用：构建 prompt → 保存 → 调用 → 保存响应。\n    参数:\n        step_name: 步骤文件夹名 (如 'step_1')\n        call_name: 本次调用名 (如 '1a_structured_extraction')\n        prompt_fn: build_*_prompt 函数\n        *args, **kwargs: 传给 prompt_fn\n        system_prompt: 系统提示词\n    返回: LLM 响应 (dict)\n    \"\"\"\n    prompt_text = prompt_fn(*args, **kwargs)\n    response = call_deepseek(prompt_text, system=system_prompt, json_mode=True)\n    save_llm_call(step_name, call_name, prompt_text, system_prompt, response, is_json=True)\n    return response\n\ndef do_text_call(step_name, call_name, prompt_fn, *args, system_prompt=\"\", **kwargs):\n    \"\"\"文本模式 LLM 调用。用于 Step 1b（返回 markdown）。\"\"\"\n    prompt_text = prompt_fn(*args, **kwargs)\n    response = call_deepseek(prompt_text, system=system_prompt, json_mode=False)\n    save_llm_call(step_name, call_name, prompt_text, system_prompt, response, is_json=False)\n    return response\n\nprint(\"辅助函数就绪\")\nprint(f\"  save_llm_call(step_name, call_name, prompt, system, response, is_json)\")\nprint(f\"  do_json_call(step_name, call_name, prompt_fn, *args, system_prompt=, **kwargs)\")\nprint(f\"  do_text_call(step_name, call_name, prompt_fn, *args, system_prompt=, **kwargs)\")"

---\n## Step 1：名称固化 + 精修模组（2 calls 并行）\n\n**Step 1a** 提取场景/NPC ID → **Step 1b** 生成精修叙事文本。两者独立，可并行。

In [ ]:
# ═══ Step 1a: 结构化提取 ═══\n# 输入: 原始模组文档\n# 输出: module_meta + scenes[{name,id}] + characters[{name,id}]\nwith ThreadPoolExecutor(max_workers=2) as ex:\n    f1a = ex.submit(do_json_call,\n        \"step_1\", \"1a_structured_extraction\",\n        build_step1a_prompt, content,\n        system_prompt=STEP1A_SYSTEM\n    )\n    f1b = ex.submit(do_text_call,\n        \"step_1\", \"1b_condensed_text\",\n        build_step1b_prompt, content,\n        system_prompt=STEP1B_SYSTEM\n    )\n    step1a = f1a.result()\n    step1b_raw = f1b.result()\n\n# Step 1b 返回的是 markdown 字符串，包裹为 dict\nstep1b = {\"condensed_text\": step1b_raw} if isinstance(step1b_raw, str) else step1b_raw\n\nscenes = step1a.get(\"scenes\", [])\ncharacters = step1a.get(\"characters\", [])\ncondensed_text = step1b.get(\"condensed_text\", \"\")\n\n# 保存 Step 1 汇总\nwith open(f\"{DEBUG_ROOT}/step_1/_summary.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump({\n        \"meta\": step1a.get(\"module_meta\", {}),\n        \"scenes\": scenes,\n        \"characters\": characters,\n        \"condensed_text_length\": len(condensed_text),\n    }, f, ensure_ascii=False, indent=2)\n\nprint(f\"Step 1a: {len(scenes)} 场景, {len(characters)} 角色\")\nfor s in scenes:\n    print(f\"  {s['id']}: {s['name']}\")\nprint(f\"Step 1b: condensed_text {len(condensed_text)} 字符\")\nprint(f\"产物: {DEBUG_ROOT}/step_1/1a_*/ 和 1b_*/\")"

In [ ]:
# 查看 condensed_text 前 600 字\nprint(condensed_text[:600])\nprint(\"...\" if len(condensed_text) > 600 else \"\")"

---\n## Step 2：内容生成\n\n**2a** interactions 先跑（固化 flag 名称） → **2b** events + auto_triggers **2c** L1 + L3 并行

In [ ]:
# ═══ Step 2a: Interactions ═══\n# 输入: condensed_text + scenes 列表\n# 输出: interactions 列表（含 ID + flag 名称 + 空 enemy_ref/weapon_ref）\nstep2a = do_json_call(\n    \"step_2\", \"2a_interactions\",\n    build_step2a_prompt, condensed_text, scenes,\n    system_prompt=STEP2A_SYSTEM\n)\ninteractions = step2a.get(\"interactions\", [])\nprint(f\"Interactions: {len(interactions)} 个\")\nfor i in interactions[:5]:\n    flags = [s.get('key','') for s in i.get('side_effects',[]) if s.get('type')=='flag_set']\n    print(f\"  {i['id']}: {i['name']} (场景 {i.get('scene','?')})\" + (f\" [flag: {flags}]\" if flags else \"\"))\nif len(interactions) > 5:\n    print(f\"  ... 共 {len(interactions)} 个\")"

In [ ]:
# ═══ Step 2b + 2c: 4 calls 并行 ═══\n# 2b: events + auto_triggers（注入 interactions 的 ID/flag）\n# 2c: L1 + L3（独立）\nwith ThreadPoolExecutor(max_workers=4) as ex:\n    f_ev = ex.submit(do_json_call,\n        \"step_2\", \"2b_events\",\n        build_step2b_events_prompt, condensed_text, scenes, interactions,\n        system_prompt=STEP2B_EVENTS_SYSTEM\n    )\n    f_at = ex.submit(do_json_call,\n        \"step_2\", \"2b_auto_triggers\",\n        build_step2b_at_prompt, condensed_text, scenes, interactions,\n        system_prompt=STEP2B_AT_SYSTEM\n    )\n    f_l1 = ex.submit(do_json_call,\n        \"step_2\", \"2c_l1\",\n        build_step2c_l1_prompt, condensed_text, scenes,\n        system_prompt=STEP2C_L1_SYSTEM\n    )\n    f_l3 = ex.submit(do_json_call,\n        \"step_2\", \"2c_l3\",\n        build_step2c_l3_prompt, condensed_text, scenes,\n        system_prompt=STEP2C_L3_SYSTEM\n    )\n    events_data = f_ev.result()\n    at_data = f_at.result()\n    l1_data = f_l1.result()\n    l3_data = f_l3.result()\n\nevents = events_data.get(\"events\", [])\nauto_triggers = at_data.get(\"auto_triggers\", [])\n\n# 保存 Step 2 汇总\nwith open(f\"{DEBUG_ROOT}/step_2/_summary.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump({\n        \"interactions_count\": len(interactions),\n        \"events_count\": len(events),\n        \"auto_triggers_count\": len(auto_triggers),\n        \"l1_scenes\": list(l1_data.keys()),\n        \"l3_world_rules\": len(l3_data.get(\"world_rules\", [])),\n    }, f, ensure_ascii=False, indent=2)\n\nprint(f\"Events: {len(events)} 个\")\nfor ev in events:\n    print(f\"  {ev.get('id','?')}: {ev.get('name','?')}\")\nprint(f\"\\nAuto-triggers: {len(auto_triggers)} 个\")\nfor at in auto_triggers:\n    print(f\"  {at.get('id','?')}: {at.get('name','?')} → {at.get('effect_type','?')} (场景 {at.get('scene','?')})\")\nprint(f\"\\nL1: {len(l1_data)} 场景\")\nprint(f\"L3: {len(l3_data.get('world_rules',[]))} 世界规则, {len(l3_data.get('scene_intents',{}))} 场景意图\")"

---\n## Step 3：依赖解析 + 交叉核对（2 calls 串行）\n\n**3a** 统一 flag 名称 + 补全 requirement 引用 → **3b** L1 ↔ L2 校对（依赖 3a 输出）

In [ ]:
# ═══ Step 3a: L2 依赖解析 ═══\n# 输入: condensed_text + Step 2 产出的 interactions, events, auto_triggers\n# 输出: 统一 flag 名 + 补全 requirement 引用\nstep3a = do_json_call(\n    \"step_3\", \"3a_dependency_resolution\",\n    build_step3a_prompt,\n    condensed_text, interactions, events, auto_triggers,\n    system_prompt=STEP3A_SYSTEM\n)\ninteractions = step3a.get(\"interactions\", interactions)\nevents = step3a.get(\"events\", events)\nauto_triggers = step3a.get(\"auto_triggers\", auto_triggers)\nprint(f\"Flag mapping: {step3a.get('flag_mapping', {})}\")\nprint(f\"依赖解析完成\")"

In [ ]:
# ═══ Step 3b: L1 ↔ L2 交叉核对 ═══\n# 输入: condensed_text + L1 + 3a 修正后的 L2 + L3 + scenes 列表\n# 输出: 修正后的 l1_data + l3_data\nl2_completed = {\n    \"interactions\": interactions,\n    \"events\": events,\n    \"auto_triggers\": auto_triggers,\n}\nstep3b = do_json_call(\n    \"step_3\", \"3b_cross_check\",\n    build_step3b_prompt,\n    condensed_text, l1_data, l2_completed, l3_data, scenes,\n    system_prompt=STEP3B_SYSTEM\n)\nl1_data = step3b.get(\"l1_data\", l1_data)\nl3_data = step3b.get(\"l3_data\", l3_data)\nprint(f\"交叉核对完成\")\nprint(f\"  L1: {len(l1_data)} 场景\")\nprint(f\"  L3 scene_intents: {list(l3_data.get('scene_intents', {}).keys())}\")"

---\n## Step 4：Library 匹配\n\n从武器/敌人库中为 interactions 和 auto_triggers 的占位符填入具体名称。

In [ ]:
# ═══ Step 4: Library 匹配 ═══\n# 输入: interactions + auto_triggers + 库列表 + 场景描述 + L3 scene_intents\n# 输出: enemy_ref / weapon_ref / effect_ref 已填入\nweapon_names = [w.name for w in wl.list_all()]\nenemy_names = [e.name for e in el.list_all()]\n\n# 构建场景描述（从 L1 数据提取）\nname_to_id = {s[\"name\"]: s[\"id\"] for s in scenes if s.get(\"name\") and s.get(\"id\")}\nl2_descriptions = {}\nfor name, sdata in l1_data.items():\n    sid = name_to_id.get(name, name)\n    desc = sdata.get(\"description\", \"\") or sdata.get(\"atmosphere\", \"\")\n    if desc:\n        l2_descriptions[sid] = desc\n\nscene_intents_for_s4 = l3_data.get(\"scene_intents\", {})\n\nif weapon_names or enemy_names:\n    step4 = do_json_call(\n        \"step_4\", \"4_library_matching\",\n        build_step4_prompt,\n        interactions, auto_triggers, l2_descriptions,\n        scene_intents_for_s4, condensed_text,\n        weapon_names, enemy_names,\n        system_prompt=STEP4_SYSTEM\n    )\n    interactions = step4.get(\"interactions\", interactions)\n    auto_triggers = step4.get(\"auto_triggers\", auto_triggers)\n    print(f\"Library 匹配完成\")\nelse:\n    print(\"Library 匹配跳过（无库可用）\")"

---\n## 最终验证 & 保存

In [ ]:
# ═══ Schema 验证 + 交叉引用 ═══\n# 按 scene ID 分组构建 L2 for validation\nscenes_by_sid = {}\nfor inter in interactions:\n    sid = inter.get(\"scene\", \"unknown\")\n    scenes_by_sid.setdefault(sid, {\n        \"interactions\": [], \"encounters\": [],\n        \"scene_weapons\": [], \"auto_triggers\": [],\n    })\n    scenes_by_sid[sid][\"interactions\"].append(inter)\nfor at in auto_triggers:\n    sid = at.get(\"scene\", \"unknown\")\n    scenes_by_sid.setdefault(sid, {\n        \"interactions\": [], \"encounters\": [],\n        \"scene_weapons\": [], \"auto_triggers\": [],\n    })\n    scenes_by_sid[sid][\"auto_triggers\"].append(at)\n\nl2_for_validation = {\n    \"scenes\": scenes_by_sid,\n    \"events\": events,\n    \"npc_profiles\": {},\n}\n\n# Schema 验证\nschema_reports = validate_all(l1_data, l2_for_validation, l3_data)\nprint(\"═══ Schema 验证 ═══\")\nfor layer, report in schema_reports.items():\n    status = \"PASS\" if report.is_valid else \"ISSUES\"\n    print(f\"  {layer} [{status}]: {report.summary()}\")\n\n# 交叉引用\ncross_ref = cross_validate_layers(l1_data, l2_for_validation, l3_data, weapon_lib=wl, enemy_lib=el)\nprint(f\"\\n═══ 交叉引用 ═══\")\nprint(f\"  {cross_ref.summary()}\")\n\n# 保存验证报告\nwith open(f\"{DEBUG_ROOT}/_validation_report.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump({\n        \"schema\": {l: {\"errors\": len(r.errors), \"warnings\": len(r.warnings), \"is_valid\": r.is_valid}\n                     for l, r in schema_reports.items()},\n        \"cross_ref\": {\"errors\": len(cross_ref.errors), \"warnings\": len(cross_ref.issues), \"is_valid\": cross_ref.is_valid},\n    }, f, ensure_ascii=False, indent=2)"

In [ ]:
# ═══ 保存最终结果到 data/modules/ ═══\nMODULE_DIR = \"../data/modules/常暗之厢\"\nos.makedirs(MODULE_DIR, exist_ok=True)\n\n# L1\nwith open(f\"{MODULE_DIR}/l1_player.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump(l1_data, f, ensure_ascii=False, indent=2)\n\n# L2\nl2_out = {\n    \"scenes\": scenes_by_sid,\n    \"events\": events,\n    \"npc_profiles\": {},\n}\nwith open(f\"{MODULE_DIR}/l2_keeper.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump(l2_out, f, ensure_ascii=False, indent=2)\n\n# L3\nwith open(f\"{MODULE_DIR}/l3_designer.json\", \"w\", encoding=\"utf-8\") as f:\n    json.dump(l3_data, f, ensure_ascii=False, indent=2)\n\nprint(f\"最终结果已保存至 {MODULE_DIR}/\")\nprint(f\"调试产物: {DEBUG_ROOT}/\")"

In [ ]:
print(\"=\" * 60)\nprint(\"四步渐进式解析完成\")\nprint(\"=\" * 60)\nprint(f\"Step 1: {len(scenes)} 场景, {len(characters)} 角色, {len(condensed_text)} 字 condensed_text\")\nprint(f\"Step 2: {len(interactions)} interactions, {len(events)} events, {len(auto_triggers)} auto_triggers\")\nprint(f\"        {len(l1_data)} L1 场景, {len(l3_data.get('world_rules',[]))} 世界规则\")\nprint(f\"Step 3: 依赖解析 + 交叉核对完成\")\nprint(f\"Step 4: Library 匹配{'完成' if weapon_names or enemy_names else '跳过'}\")\nprint(f\"\")\nprint(f\"总 LLM 调用: 10 (Step 1:2 + Step 2:5 + Step 3:2 + Step 4:1)\")\nprint(f\"调试产物: {DEBUG_ROOT}/\")\nprint(f\"├── step_1/   (1a_structured_extraction, 1b_condensed_text)\")\nprint(f\"├── step_2/   (2a_interactions, 2b_events, 2b_auto_triggers, 2c_l1, 2c_l3)\")\nprint(f\"├── step_3/   (3a_dependency_resolution, 3b_cross_check)\")\nprint(f"  step_4/   (4_library_matching)")
print(f\"\")\nprint(f\"最终模组: {MODULE_DIR}/\")\nprint(f\"  l1_player.json, l2_keeper.json, l3_designer.json\")\nprint(f\"状态: {'PASS' if cross_ref.is_valid else 'HAS_ISSUES'}\")"